# Data Ingestion
In this section, we will learn about storing the document chunks into the vector Db, and build a RAG database.

In [ ]:
# Install packages
!pip install langchain langchain_core langchain_community langchain_openai langchain_huggingface chromadb faiss-cpu pymupdf pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.6/326.6 kB 23.6 MB/s eta 0:00:00


 17%|█▋        | 1/6 [10:11<50:57, 611.58s/it]


In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


Mounted at /content/drive


In [ ]:
import shutil
import os

persist_dir = "/content/drive/MyDrive/edurekaai/_data/rag/chroma_store"
persist_dir_temp = "/content/chroma_store"

# 💣 Remove the old Chroma store if it exists
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)
    print("🧹 Old Chroma store deleted — starting fresh!")



🧹 Old Chroma store deleted — starting fresh!


In [ ]:
# Building Chroma dB (using texts input files)
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# Step 1: Load documents from directory
loader = DirectoryLoader(
    '/content/drive/MyDrive/edurekaai/_data/rag/texts',
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

# Step 2: Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.split_documents(documents)

print(f"✅ Loaded {len(documents)} documents.")
print(f"✅ Split into {len(chunks)} chunks.\n")

# Step 3: Initialize embeddings (Hugging Face model)
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Step 4: Store chunks in ChromaDB (persistent)
chroma_db = Chroma.from_documents(
    documents=chunks,
    embedding=hf_embeddings,
    persist_directory=persist_dir   # You can change this path
)

print(f"Vector store created with {chroma_db._collection.count()} vectors.")

# Save to disk
chroma_db.persist()
print("✅ All chunks and embeddings stored in ChromaDB successfully!")

100%|██████████| 3/3 [00:02<00:00,  1.11it/s]


✅ Loaded 3 documents.
✅ Split into 19 chunks.



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created with 19 vectors.
✅ All chunks and embeddings stored in ChromaDB successfully!


/tmp/ipython-input-1214732323.py:41: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma_db.persist()


In [ ]:
# !rm -rdf /content/chroma_store/*
# !cp -r /content/chroma_store /content/drive/MyDrive/edurekaai/_data/rag/chroma_store


## Simillarity Search
It uses vector dot_prouct to find simillar vectors. The vectors that returns the maximum dot_prodct value.

In [ ]:
# --- Optional: Test retrieval ---
query = "What are the type of machine learning fields?"
results = chroma_db.similarity_search(query, k=3)
# results = chroma_db.similarity_search_with_score(query, k=3)
print("\n🔍 Top 3 matching chunks:")
# results
for i, doc in enumerate(results):
    print(f"\n[{i+1}] Content: {doc.page_content[:100]}...")
    print(f"\n[{i+1}] Metadata: {doc.metadata}...")
    # print(f"\n[{i+1}] Simillarity Score: {doc}...")
    print(f"=================================================================")


🔍 Top 3 matching chunks:

[1] Content: ---

## 🧩 Types of Machine Learning

Machine Learning can be categorized into three main types:

###...

[1] Metadata: {'source': '/content/drive/MyDrive/edurekaai/_data/rag/texts/machine_learning.txt'}...

[2] Content: # 🤖 Introduction to Machine Learning

Machine Learning (ML) is a branch of **Artificial Intelligence...

[2] Metadata: {'source': '/content/drive/MyDrive/edurekaai/_data/rag/texts/machine_learning.txt'}...

[3] Content: Examples:
- Game-playing AI (like AlphaGo)
- Robotics and autonomous driving

Popular algorithms: **...

[3] Metadata: {'source': '/content/drive/MyDrive/edurekaai/_data/rag/texts/machine_learning.txt'}...


In [ ]:
# --- Optional: Test retrieval ---
query = "What are the type of machine learning fields?"
results = chroma_db.similarity_search_with_score(query, k=3)
print("\n🔍 Top 3 matching chunks:")
# results
for i, doc in enumerate(results):
    print(f"\n[{i+1}] Content: {doc[0].page_content[:200]}...")
    print(f"\n[{i+1}] Metadata: {doc[0].metadata}...")
    print(f"\n[{i+1}] Simillarity Score: {doc[1]}...")
    print(f"=================================================================")


🔍 Top 3 matching chunks:

[1] Content: ---

## 🧩 Types of Machine Learning

Machine Learning can be categorized into three main types:

### 1. **Supervised Learning**
The model is trained on labeled data — meaning the correct answer is kno...

[1] Metadata: {'source': '/content/drive/MyDrive/edurekaai/_data/rag/texts/machine_learning.txt'}...

[1] Simillarity Score: 0.7074421644210815...

[2] Content: # 🤖 Introduction to Machine Learning

Machine Learning (ML) is a branch of **Artificial Intelligence (AI)** that enables systems to learn and improve automatically from experience without being explic...

[2] Metadata: {'source': '/content/drive/MyDrive/edurekaai/_data/rag/texts/machine_learning.txt'}...

[2] Simillarity Score: 0.726955771446228...

[3] Content: Examples:
- Game-playing AI (like AlphaGo)
- Robotics and autonomous driving

Popular algorithms: **Q-Learning, Deep Q-Network (DQN)**

---

## 💡 How Machine Learning Works

The general workflow of an...

[3] Metadata: {'source':

In [ ]:
# Initialize LLM
from langchain_openai import ChatOpenAI
from langchain.schema.output_parser import StrOutputParser

llm = ChatOpenAI(temperature=0,model_name="gpt-3.5-turbo") | StrOutputParser()

#test LLM
response = llm.invoke("What are the type of machine learning fields?")
response
#

'1. Supervised Learning\n2. Unsupervised Learning\n3. Semi-supervised Learning\n4. Reinforcement Learning\n5. Deep Learning\n6. Transfer Learning\n7. Online Learning\n8. Ensemble Learning\n9. Bayesian Learning\n10. Inductive Learning'

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

# Assume you have:
# chroma_db = Chroma.from_documents(...)
# llm = your LLM instance

retriever = chroma_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Define prompt template
prompt_template = """
You are a helpful AI assistant for Q&A tasks.
Use the following context to answer the question at the end and use max 3 sentences to keep it concise.
If you don't find the answer in the context provided, just say you don't know.

Context: {context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

# LCEL Retrieval Chain
chain = (
    {
        "context": retriever,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)


# Run query
response = chain.invoke("where neural network is used?")
print(response)



Neural networks are used in various applications such as speech recognition (e.g., Siri, Alexa), image recognition (e.g., Google Photos, facial recognition), natural language processing (e.g., ChatGPT, translation apps), autonomous vehicles, and medical image analysis.


## Max Marginal Relevance (MMR)
Similarity search returns the top-k results that are closest to your query vector. It uses dot_prodct to find the close matched vectors. This sometimes could lead to strictly relevant passages.

MMR returns result that are relevant to the query, but diverse from each other. It reduces redundancy and covers more topics / perspectives. It is great for QA, RAG, and summerization.

In [ ]:
# --- Optional: Test retrieval ---
query = "What are the type of machine learning fields?"
results = chroma_db.max_marginal_relevance_search(query, k=3)
print("\n🔍 Top 3 matching chunks:")
# results
for i, doc in enumerate(results):
    print(f"\n[{i+1}] Content: {doc.page_content[:100]}...")
    print(f"\n[{i+1}] Metadata: {doc.metadata}...")
    print(f"=================================================================")


🔍 Top 3 matching chunks:

[1] Content: ---

## 🧩 Types of Machine Learning

Machine Learning can be categorized into three main types:

###...

[1] Metadata: {'source': '/content/drive/MyDrive/edurekaai/_data/rag/texts/machine_learning.txt'}...

[2] Content: ---

## 🧰 Best Practices in ML Development

- Collect **quality data**, not just large data.  
- Avo...

[2] Metadata: {'source': '/content/drive/MyDrive/edurekaai/_data/rag/texts/machine_learning.txt'}...

[3] Content: # Compile and train
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accur...

[3] Metadata: {'source': '/content/drive/MyDrive/edurekaai/_data/rag/texts/deep_learning.txt'}...


## Build faiss Vector dB

In [ ]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain.vectorstores import FAISS
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA
from langchain.schema.output_parser import StrOutputParser
import os



In [ ]:
llm = OpenAI(temperature=0.7)
openai_embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", disallowed_special=())

In [ ]:
# Step 1: Load documents from directory
loader = DirectoryLoader(
    '/content/drive/MyDrive/edurekaai/_data/rag/pdfs',
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)
documents = loader.load()

100%|██████████| 6/6 [00:06<00:00,  1.11s/it]


In [ ]:
# Step 2: Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.split_documents(documents)

print(f"✅ Loaded {len(documents)} documents.")
print(f"✅ Split into {len(chunks)} chunks.\n")



✅ Loaded 239 documents.
✅ Split into 980 chunks.



In [ ]:
# Step 3: Store chunks in faiss (stores it in memory)
faiss_db = FAISS.from_documents(documents=chunks, embedding=openai_embeddings)

In [ ]:
# Step 4: Create a Retriever
retriever = faiss_db.as_retriever()

query= "what are the key take away from the documents?"

docs = retriever.get_relevant_documents(query)

combined_text="\n".join([doc.page_content for doc in docs])
print(combined_text)


5 
 
Research a Relevance Content addresses the research question 
directly. 
 b Depth of analysis Demonstrates critical thinking and 
synthesis of multiple sources. 
 c Accuracy Facts, citations, and interpretations are 
correct. 
 d Originality Offers fresh insights or perspectives, not 
just generic summaries. 
 e Clarity and structure Logical flow and academic tone. 
 f Evidence use Quality and integration of supporting 
references. 
Ethics a Bias avoidance No stereotyping, discriminatory 
assumptions, or harmful framing. 
 b Respectfulness Tone is professional, inclusive, and 
culturally sensitive. 
 c Transparency Clear about limitations, uncertainties, and 
the model’s role. 
 d Fairness Balanced treatment of perspectives where 
appropriate. 
 e Harm minimization Avoids potentially damaging advice or 
implications. 
 f Ethical awareness Recognizes and addresses moral/ethical 
dimensions of the task. 
 
2.4 Statistical analysis
system-level interventions (such as monitoring and p

In [ ]:
# Option 1
# Create chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

# Invoke chain
response = qa_chain.invoke(query)
response['result']

'\nThe key takeaways from the documents are:\n- The importance of relevance, depth of analysis, accuracy, originality, clarity and structure, and evidence use in research\n- The need for ethical considerations, such as bias avoidance, respectfulness, transparency, fairness, harm minimization, and ethical awareness, in research\n- The limitations and brittleness of mitigations and processes in altering GPT-4\'s behavior and preventing misuse\n- The potential for disturbing or offensive content in the document, including sexual, hateful, or violent content\n- The use of a rating system from 1 (strongly disagree) to 5 (strongly agree) to evaluate the criteria in different domains, such as education, research, and ethics\n- The importance of alignment with learning objectives, structure and sequencing, and content quality and accuracy in lesson plans\n- The limitations of GPT-4 for cybersecurity operations due to its "hallucination" tendency and limited context window. '

In [ ]:
# Option 2
# Prompt Templates
template = f"Based on the following context {combined_text}, answer the user queries: {query}"
prompt = PromptTemplate(
    input_variables=["query", "combined_text"],
    template=template
)

formatted_prompt = prompt.format(query=query, combined_text=combined_text)
response = llm.predict(formatted_prompt)
response

"\nBased on the context, here are some key takeaways from the documents:\n\n1. The document discusses the development and use of GPT-4, a powerful language model. \n2. Various domains, such as education, research, and ethics, were used to evaluate the model's performance. \n3. The model's limitations and potential for misuse were also discussed. \n4. Content warning was provided as the document contains potentially offensive or disturbing content. \n5. The limitations of GPT-4 in the cybersecurity field were also mentioned."

In [ ]:
def continual_chat():
  print("Starting the chat with AI! Type 'exit' to end the coversation")
  chat_history = [] #collection of chat history
  while True:
    query = input("You: ")
    if query.lower() == "exit":
      break

    #process user query
    result = qa_chain({"query": query})
    print(f"AI: {result['result']}")

    #update chat history
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=result['result']))

In [ ]:
continual_chat()

Starting the chat with AI! Type 'exit' to end the coversation
You: what is data factory?
AI:  I'm sorry, I don't have enough information to answer that question.
You: what do u know?
AI:  I am a large language model trained by OpenAI and I know a lot about various topics, but I am not a human and I am not capable of having experiences or memories like a human. 
You: tell me about llm
AI:  Large language models, also known as LLMs, have become an increasingly prevalent part of our day-to-day lives, with their use extending to a wide range of domains including web browsing, voice assistants, and coding assistance tools. They have the potential to significantly impact society in numerous ways and are constantly being developed and improved upon. Some examples of LLMs include GPT-4 and GPT-5, which have shown advancements in reasoning, multimodality, and task generalization. LLMs are also being used in various fields such as education, clinical practice, research, and ethics. They are cons